# Notebook 4: RAG with Static Demonstrations

**Who this is for:** Digital Humanities students with no prior AI or programming background.

**What you will learn**
1. Why a language model cannot answer questions about documents it has never seen
2. What **RAG** means: Retrieve, Augment, Generate
3. How to put sources into a prompt, and ask the model to cite them
4. How to make the model say "the sources do not say" instead of guessing
5. How this works in Arabic and across languages

**Before you start:** you need an API key for Claude, OpenAI or Gemini. Notebook 3 (section 0) explains how to get one. Lines that start with `#` are comments for humans.

## 0. Setup

In [ ]:
# Install the libraries for the three providers. "-q" keeps the output short.
!pip install -q anthropic openai google-genai pandas

In [ ]:
import os
from getpass import getpass

# Choose ONE provider. You only need a key for the one you choose (see Notebook 3, section 0).
PROVIDER = "claude"        # "claude", "openai" or "gemini"

# Model names are just settings. Change them if a name is retired (see each provider's documentation).
CLAUDE_MODEL = "claude-sonnet-4-5"
OPENAI_MODEL = "gpt-4o-mini"
GEMINI_MODEL = "gemini-2.5-flash"

# The name under which the key is saved in Colab Secrets (the key icon in the left sidebar).
KEY_NAMES = {"claude": "ANTHROPIC_API_KEY", "openai": "OPENAI_API_KEY", "gemini": "GEMINI_API_KEY"}

def get_key(name):
    """Read the key from Colab Secrets; if it is missing, ask for it (the typing is hidden)."""
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name) or getpass(f"Enter {name}: ")

API_KEY = get_key(KEY_NAMES[PROVIDER])
print("Key found:", bool(API_KEY))      # prints True or False, never the key itself

In [ ]:
def ask(prompt, system=None, max_tokens=500):
    """Send a prompt to the chosen provider and return the answer as text."""

    if PROVIDER == "claude":
        import anthropic
        client = anthropic.Anthropic(api_key=API_KEY)
        options = dict(model=CLAUDE_MODEL, max_tokens=max_tokens, temperature=0,
                       messages=[{"role": "user", "content": prompt}])
        if system:
            options["system"] = system
        return client.messages.create(**options).content[0].text

    if PROVIDER == "openai":
        from openai import OpenAI
        client = OpenAI(api_key=API_KEY)
        messages = [{"role": "system", "content": system}] if system else []
        messages.append({"role": "user", "content": prompt})
        reply = client.chat.completions.create(model=OPENAI_MODEL, messages=messages,
                                               temperature=0, max_tokens=max_tokens)
        return reply.choices[0].message.content

    if PROVIDER == "gemini":
        from google import genai
        from google.genai import types
        client = genai.Client(api_key=API_KEY)
        config = types.GenerateContentConfig(system_instruction=system, temperature=0,
                                             max_output_tokens=max_tokens)
        return client.models.generate_content(model=GEMINI_MODEL, contents=prompt, config=config).text

    raise ValueError("PROVIDER must be 'claude', 'openai' or 'gemini'")

# Quick test: the answer should be the word "ready".
print(ask("Reply with one word: ready", max_tokens=10))

> Some newer models fix the temperature or ignore it. If you see an error mentioning `temperature`, delete `temperature=0` from the code above.

## 1. The problem: the model has never seen your documents

To test this honestly we need a document that **no model has ever seen**. So we invented a small archive: the records of the *Cedar Valley Reading Society*, a made-up reading club from the 1920s and 1930s. It has 12 short records, each in English and in Arabic.

**Everything in this archive is invented for the exercise.**

In [ ]:
#@title Load the invented Cedar Valley archive (double-click to show the code) { display-mode: "form" }
ARCHIVE = [
    {"id": "r01", "title": "Founding",
     "en": "The Cedar Valley Reading Society was founded in 1921 by the teacher Salim Haddad in a rented room above a bakery.",
     "ar": "تأسست جمعية وادي الأرز للقراءة عام 1921 على يد المعلم سليم حداد في غرفة مستأجرة فوق مخبز."},
    {"id": "r02", "title": "Library size",
     "en": "In 1930 the library of the society held 1,200 volumes, mostly Arabic poetry, history books and translated novels.",
     "ar": "في عام 1930 ضمّت مكتبة الجمعية 1200 مجلد، معظمها دواوين شعر عربي وكتب تاريخ وروايات مترجمة."},
    {"id": "r03", "title": "Members in 1928",
     "en": "The lending register for 1928 lists 340 members. Each member could borrow two books for fourteen days.",
     "ar": "يسجّل دفتر الإعارة لعام 1928 أسماء 340 عضوًا. وكان يحق لكل عضو استعارة كتابين لمدة أربعة عشر يومًا."},
    {"id": "r04", "title": "Late fines",
     "en": "A book returned late cost the borrower a fine of one piaster per day. The money was used to repair damaged books.",
     "ar": "كانت غرامة تأخير إعادة الكتاب قرشًا واحدًا عن كل يوم، وكان المال يُستخدم لإصلاح الكتب التالفة."},
    {"id": "r05", "title": "New building",
     "en": "In 1936 the society moved to a new building near the old market. The move was paid for by donations from local merchants.",
     "ar": "في عام 1936 انتقلت الجمعية إلى مبنى جديد قرب السوق القديم، ودُفعت تكاليف الانتقال من تبرعات تجار المدينة."},
    {"id": "r06", "title": "Monthly bulletin",
     "en": "From 1925 to 1939 the society published a monthly bulletin. Only 61 issues survive today, and they are kept in the municipal archive.",
     "ar": "من عام 1925 إلى عام 1939 أصدرت الجمعية نشرة شهرية. لم يبق منها اليوم سوى 61 عددًا، وهي محفوظة في أرشيف البلدية."},
    {"id": "r07", "title": "Women's reading circle",
     "en": "In 1931 the society started a weekly evening reading circle for women, led by the teacher Nadia Khoury.",
     "ar": "في عام 1931 بدأت الجمعية حلقة قراءة مسائية أسبوعية للنساء، أدارتها المعلمة نادية خوري."},
    {"id": "r08", "title": "Poetry lecture",
     "en": "In 1927 the visiting poet Khalil Saba gave a lecture on modern Arabic poetry, and about 200 people attended.",
     "ar": "في عام 1927 ألقى الشاعر الزائر خليل سابا محاضرة عن الشعر العربي الحديث، وحضرها نحو 200 شخص."},
    {"id": "r09", "title": "Merchant's donation",
     "en": "In 1933 the merchant Yusuf Kassab donated 300 books from his private library, including a rare copy of a seventeenth-century geography manuscript.",
     "ar": "في عام 1933 تبرّع التاجر يوسف كسّاب بـ 300 كتاب من مكتبته الخاصة، من بينها نسخة نادرة من مخطوط جغرافي يعود إلى القرن السابع عشر."},
    {"id": "r10", "title": "Closure",
     "en": "The society closed in 1951 when its funds ran out. Its books were kept in a school storeroom, and about 900 volumes were catalogued again in 1990.",
     "ar": "أُغلقت الجمعية عام 1951 بعد نفاد أموالها. حُفظت كتبها في مخزن مدرسة، وأُعيد فهرسة نحو 900 مجلد منها عام 1990."},
    {"id": "r11", "title": "Digitisation",
     "en": "In 2019 a university team digitised the surviving bulletins and the 1928 lending register. The scans can be read online for free.",
     "ar": "في عام 2019 رقمن فريق جامعي النشرات الباقية ودفتر الإعارة لعام 1928، ويمكن قراءة النسخ الممسوحة مجانًا على الإنترنت."},
    {"id": "r12", "title": "Reading habits",
     "en": "The 1928 register shows that poetry was the most borrowed category, followed by history and then translated novels.",
     "ar": "يُظهر سجل عام 1928 أن الشعر كان أكثر الأصناف استعارة، يليه التاريخ ثم الروايات المترجمة."},
]

In [ ]:
# A dictionary lets us find a record by its id, for example by_id["r03"].
by_id = {r["id"]: r for r in ARCHIVE}

# Show the list of records (titles only).
for r in ARCHIVE:
    print(r["id"], "-", r["title"])

In [ ]:
# Ask the model about the society WITHOUT giving it any sources.
question = "How many members did the Cedar Valley Reading Society have in 1928?"
print(ask(question))

The model cannot know the answer. A good model says it does not know. A weaker one may **invent** a number. Either way, it cannot help with your documents unless you give them to it.

## 2. The RAG idea

**RAG** stands for **Retrieval-Augmented Generation**. It has three steps:

```
Question --> 1. RETRIEVE the relevant passages
         --> 2. AUGMENT the prompt: put the passages in it
         --> 3. GENERATE the answer from the passages
```

In this notebook **you** do step 1 by hand (we call this *static*: you choose the passages yourself). In Notebook 5, a vector database does it automatically.

## 3. Augment: put the sources in the prompt

In [ ]:
def build_prompt(question, records, lang="en"):
    """Put the chosen records into the prompt as numbered sources, then add the question."""
    sources = ""
    for number, record in enumerate(records, start=1):
        sources += f"[{number}] {record[lang]}\n"          # for example: [1] The lending register...

    return (
        "Answer the question using ONLY the sources below.\n"
        "Cite the numbers of the sources you used, for example [1].\n"
        "If the sources do not contain the answer, say exactly: The sources do not say.\n"
        "Answer in the same language as the question.\n\n"
        f"Sources:\n{sources}\n"
        f"Question: {question}"
    )

# Step 1 (by hand): we choose record r03, the lending register of 1928.
chosen = [by_id["r03"]]

prompt = build_prompt(question, chosen)
print(prompt)

In [ ]:
# Step 3: send the prompt and read the answer.
print(ask(prompt))

The answer now comes from our record, and the model points to the source number. Read the prompt above once more: **there is no magic**. RAG is simply a prompt that contains the right text.

Now a question that needs **two** records. We choose both by hand.

In [ ]:
question2 = "Which category of books was borrowed most in 1928, and how many members did the register list?"
chosen = [by_id["r03"], by_id["r12"]]               # members (r03) and reading habits (r12)
print(ask(build_prompt(question2, chosen)))

## 4. When the sources do not contain the answer

Our instruction tells the model what to do when the answer is missing. This is very important for research: **a missing answer should look like a missing answer**, not like a guess.

In [ ]:
# Nothing in r01 or r02 tells us who the first librarian was.
question3 = "Who was the first librarian of the society?"
chosen = [by_id["r01"], by_id["r02"]]
print(ask(build_prompt(question3, chosen)))

## 5. In Arabic, and across languages

In [ ]:
# Arabic question, Arabic source. ("What was the fine for returning a book late?")
arabic_question = "كم كانت غرامة تأخير إعادة الكتاب؟"
print(ask(build_prompt(arabic_question, [by_id["r04"]], lang="ar")))

In [ ]:
# Arabic question, ENGLISH source (lang="en"). The model reads English and answers in Arabic.
print(ask(build_prompt(arabic_question, [by_id["r04"]], lang="en")))

## 6. What happens if we choose the wrong source?

If the wrong passage reaches the model, even a good model cannot answer. This is why **retrieval quality matters**, and why Notebook 5 needs a good search.

In [ ]:
# We ask about fines but give the record about the new building (r05).
print(ask(build_prompt("What was the fine for returning a book late?", [by_id["r05"]])))

## Key takeaways

- A model does not know your documents. **Give them to it in the prompt.**
- RAG = **Retrieve** passages, **Augment** the prompt, **Generate** the answer.
- Ask the model to **cite sources** and to say **"the sources do not say"** when needed.
- The answer is only as good as the passages you retrieve.

## Exercises

1. Replace one record with a paragraph from your own research. Ask a question about it.
2. Ask: `"What did the society do for women?"` and choose the right record yourself (look at the list in section 1).
3. Change the instruction to `"Answer in one sentence."` What changes?
4. Give the model two sources that **disagree** (edit one number). What does it do?

**Next:** Notebook 5 replaces the by-hand step 1 with a vector database.